# Dars 64 — Uyga vazifa: Transformer va Attention
Har bo'limda: kod → natija → `# Xulosa:`. Google Colab'da yuqoridan pastga **Runtime → Run all** qiling.

## 1-topshiriq. O'xshashlik va softmax

In [ ]:
import numpy as np

# planshet, chegirma, daftar, qalam
names = ["planshet", "chegirma", "daftar", "qalam"]
X = np.array([[2, 1, 1],
              [0, 3, 0],
              [0, 1, 2],
              [0, 1, 3]])

# Oson: planshet va chegirma skalyar ko'paytmasi = 2*0 + 1*3 + 1*0
print("planshet · chegirma =", X[0] @ X[1])

# O'rta: S = X @ X.T
S = X @ X.T
print(S)
assert list(S[2]) == [3, 3, 5, 7]

def softmax(s):
    e = np.exp(s - s.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

w = softmax(S[2])
print("softmax(daftar):", w.round(3), "yig'indi =", w.sum())
assert np.isclose(w.sum(), 1)

# Taxmin: daftar ga eng o'xshash BOSHQA so'z -> qalam (7 ball)
S_other = S[2].astype(float).copy()
S_other[2] = -np.inf          # o'zini hisobga olmaymiz
print("Eng o'xshash:", names[int(S_other.argmax())])

# Qiyin: softmax(S[0]) va softmax(S[0]*10)
print(softmax(S[0]).round(4))
print(softmax(S[0] * 10).round(4))

# Xulosa: ballar 10 ga ko'paysa, farqlar kattalashadi va exp ularni keskin kuchaytiradi,
# shuning uchun vazn deyarli bitta (eng katta) so'zga o'tib ketadi.

## 2-topshiriq. Q, K, V qo'lda

In [ ]:
# xususiyat: [arzon, planshet, telefon]
q = np.array([1, 2, 0])          # so'rov: arzon planshet
K = np.array([[1, 1, 0],         # arzon planshet
              [0, 1, 0],         # planshet
              [1, 0, 1],         # arzon telefon
              [0, 0, 1]])        # telefon
V = np.array([5, 11, 3, 14])     # narx, mln so'm

def run(q):
    scores = K @ q
    w = softmax(scores)
    out = w @ V
    print("q =", q, "| scores =", scores, "| w =", w.round(3), "| natija =", round(float(out), 3))
    return w, out

w, out = run(q)
assert 6 < out < 7

# Xulosa: attention vaznli o'rtacha beradi, vaznlar musbat va yig'indisi 1,
# shuning uchun natija doim eng kichik va eng katta qiymat orasida bo'ladi.

# q = [0,0,1]
w, out = run(np.array([0, 0, 1]))
print("Eng katta vazn: arzon telefon va telefon (ikkalasi teng, ~0.366)")
# Xulosa: telefon xususiyati bor ikki mahsulot teng vazn oladi, natija ~8.37 ga ko'tariladi.

# Qiyin: natija 12 dan katta bo'lsin
w, out = run(np.array([-3, 0, 5]))
assert out > 12
# Xulosa: "telefon"ga katta musbat, "arzon"ga manfiy vazn berdik: arzon telefon (3) ballari tushdi,
# telefon (14) deyarli butun vaznni oldi.

## 3-topshiriq. Attention funksiyasi va √d_k

In [ ]:
def attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)
    weights = softmax(scores)
    return weights @ V, weights

X3 = np.array([[1, 0], [0, 2], [1, 1], [2, 1]])
out, w = attention(X3, X3, X3)
print("w.shape =", w.shape, "| out.shape =", out.shape)   # (4,4) va (4,2)
assert np.allclose(w.sum(axis=1), 1)

# Tajriba
np.random.seed(0)
rows = []
for d in [4, 64, 512]:
    Q = np.random.randn(8, d)
    K_ = np.random.randn(8, d)
    w1 = softmax(Q @ K_.T)                 # a) √d ga bo'lmasdan
    w2 = softmax(Q @ K_.T / np.sqrt(d))    # b) √d ga bo'lib
    rows.append((d, w1.max(axis=1).mean(), w2.max(axis=1).mean()))

print(f"{'d_k':>5} | {'√d_k siz':>9} | {'√d_k bilan':>10}")
for d, a, b in rows:
    print(f"{d:>5} | {a:>9.3f} | {b:>10.3f}")

# Xulosa: Q va K elementlari mustaqil bo'lsa, Q@K.T ballarining dispersiyasi ~d_k ga teng,
# ya'ni d_k oshsa ballar tarqoq (katta) bo'ladi va softmax deyarli one-hot (bitta so'z) beradi.
# √d_k ga bo'lish dispersiyani ~1 ga qaytaradi, vaznlar yumshoq qoladi.

## 4-topshiriq. SelfAttention moduli (PyTorch)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelfAttention(nn.Module):
    def __init__(self, C, causal=False):
        super().__init__()
        self.q = nn.Linear(C, C, bias=False)
        self.k = nn.Linear(C, C, bias=False)
        self.v = nn.Linear(C, C, bias=False)
        self.causal = causal

    def forward(self, x):                       # x: (B, T, C)
        Q, K, V = self.q(x), self.k(x), self.v(x)           # (B, T, C)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.size(-1))   # (B, T, T)
        if self.causal:
            T = x.size(1)
            mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
            scores = scores.masked_fill(~mask, float("-inf"))
        w = F.softmax(scores, dim=-1)                       # (B, T, T)
        return w @ V, w                                     # (B, T, C), (B, T, T)

torch.manual_seed(0)
sa = SelfAttention(16)
x = torch.randn(3, 6, 16)
out, w = sa(x)
print("out:", tuple(out.shape), "| w:", tuple(w.shape), "| q.weight:", tuple(sa.q.weight.shape))
# Taxmin: out (3,6,16), w (3,6,6), q.weight (16,16)
assert torch.allclose(w.sum(dim=-1), torch.ones(3, 6))

# Gradient
loss = out.pow(2).mean()
loss.backward()
print("q.weight.grad shape:", tuple(sa.q.weight.grad.shape))   # (16,16)
# Xulosa: Q ballar orqali vaznlarga, vaznlar esa out va loss ga ta'sir qiladi,
# shuning uchun zanjir qoidasi bo'yicha Wq ga ham gradient keladi.

# Parametrlar: 3 ta Linear * (16*16) = 768
n_params = sum(p.numel() for p in sa.parameters())
print("Parametrlar:", n_params)
assert n_params == 3 * 16 * 16

# 3 ta xato (asl kod):
#   scores = Q @ K / math.sqrt(Q.size(0))
#   w = F.softmax(scores, dim=0)
# 1) Q @ K  -> K ni transpozitsiya qilish kerak: Q @ K.transpose(-2, -1)
# 2) Q.size(0) batch o'lchami; d_k uchun oxirgi o'lcham kerak: Q.size(-1)
# 3) dim=0 batch bo'yicha normallaydi; kalitlar bo'yicha (oxirgi o'q) kerak: dim=-1
# To'g'ri variant:
#   scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.size(-1))
#   w = F.softmax(scores, dim=-1)
#   return w @ V, w

`attn.py` fayli kerak bo'lsa, yuqoridagi `SelfAttention` klassini shu nomli faylga ko'chirib qo'ying (Colab'da `%%writefile attn.py` bilan ham bo'ladi).

## 5-topshiriq. Causal mask
> Hujjatda 5- va 6-topshiriq matni yo'q edi. Quyidagisi standart ta'rifga asoslangan; haqiqiy matn bilan solishtirib ko'ring.

In [ ]:
torch.manual_seed(0)
sa_c = SelfAttention(16, causal=True)
x = torch.randn(2, 5, 16)
out, w = sa_c(x)

print(w[0].detach().round(decimals=2))
assert torch.allclose(w.sum(dim=-1), torch.ones(2, 5))
assert torch.all(torch.triu(w, diagonal=1) == 0)     # kelajakka vazn = 0

# Kelajakni o'zgartirsak, oldingi pozitsiyalar chiqishi o'zgarmasligi kerak
x2 = x.clone()
x2[:, -1] = torch.randn(2, 16)
out2, _ = sa_c(x2)
assert torch.allclose(out[:, :-1], out2[:, :-1], atol=1e-6)
print("Causal tekshiruvlar o'tdi")

# Xulosa: ballar softmax'dan OLDIN kelajak joylarda -inf qilinadi, shunda exp(-inf)=0 va
# har bir token faqat o'zidan oldingi tokenlarga qaraydi (matn generatsiyasida "aldash" yo'q).

## 6-topshiriq. Multi-head attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, C, n_heads, causal=False):
        super().__init__()
        assert C % n_heads == 0, "C n_heads ga bo'linishi kerak"
        self.h, self.d = n_heads, C // n_heads
        self.qkv = nn.Linear(C, 3 * C, bias=False)
        self.proj = nn.Linear(C, C, bias=False)
        self.causal = causal

    def forward(self, x):                                    # (B, T, C)
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # (B, T, C) -> (B, h, T, d)
        q, k, v = (t.view(B, T, self.h, self.d).transpose(1, 2) for t in (q, k, v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d)  # (B, h, T, T)
        if self.causal:
            mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
            scores = scores.masked_fill(~mask, float("-inf"))
        w = F.softmax(scores, dim=-1)
        out = (w @ v).transpose(1, 2).contiguous().view(B, T, C)   # boshlarni birlashtirish
        return self.proj(out), w

torch.manual_seed(0)
mha = MultiHeadAttention(16, n_heads=4)
x = torch.randn(3, 6, 16)
out, w = mha(x)
print("out:", tuple(out.shape), "| w:", tuple(w.shape))     # (3,6,16), (3,4,6,6)
assert torch.allclose(w.sum(dim=-1), torch.ones(3, 4, 6))
print("Parametrlar:", sum(p.numel() for p in mha.parameters()))   # 3*256 + 256 = 1024

# Xulosa: har bosh C ni h bo'lakka bo'lib, o'z kichik fazosida attention qiladi; shu tariqa
# turli bosh turli xil munosabatlarni (masalan, yaqin so'z, sintaksis) o'rganishi mumkin.

## Bonus. Gap uchun attention xaritasi
> Embedding'lar tasodifiy (o'qitilmagan), shuning uchun xarita faqat mexanizmni ko'rsatadi, ma'noni emas.

In [ ]:
import matplotlib.pyplot as plt

tokens = ["Uzum", "da", "arzon", "planshet", "qidirdim"]
torch.manual_seed(1)
emb = nn.Embedding(len(tokens), 16)
x = emb(torch.arange(len(tokens))).unsqueeze(0)       # (1, 5, 16)
_, w = SelfAttention(16)(x)
W = w[0].detach().numpy()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(W, cmap="viridis")
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45)
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
ax.set_xlabel("Key (qarayotgan joy)"); ax.set_ylabel("Query (so'rovchi so'z)")
fig.colorbar(im); plt.tight_layout(); plt.show()